# Segmentation mask — candidate rules

**Purpose.** The supervisor said the camera-direction idea is not
worth it, and suggested a rule for the vegetation area
closest to each observation instead; he called it a central point of
the work and said to test several approaches (see
[`docs/meetings/2026-11-04.md`](../../docs/meetings/2026-11-04.md#resposta-dos-orientadores-por-mensagem)).
No rule is chosen here: each candidate is tried on a small sample and
exported to GeoPackage to compare visually in QGIS before deciding.

**Candidates** (none grounded in a paper that does exactly this; see
[`thesis/notes.md`](../../thesis/notes.md#máscara-de-segmentação-sem-a-direção-da-câmara-testar-várias-abordagens)
for the literature search):
1. **COS 2023 polygon:** if the point falls inside a "Florestas de
   acácias" (`5.1.1.7`) polygon, the mask is that polygon.
2. **Region growing on TESSERA embeddings:** from the pixel's
   embedding, grow to neighbours above a similarity threshold, up to
   a maximum radius.
3. **Superpixels:** segment a window around the point (SLIC) and take
   the superpixel the point falls in.

**Needs** the cleaned file from `02_citizen_science/acacia_observations.ipynb`.

## 0. Setup and sample

A fixed random sample of 30 presences, to prototype quickly (not the
full 3 000); the final rule, once chosen, is applied to all of them.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
COS = ROOT / "data" / "interim" / "spatial" / "COS2023v1-S2.gpkg"
OUT = ROOT / "data" / "interim" / "spatial" / "segmentation_mask_sample.gpkg"

acacia = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
acacia = gpd.GeoDataFrame(acacia, geometry=gpd.points_from_xy(acacia.longitude, acacia.latitude), crs=4326).to_crs(3763)
sample = acacia.sample(30, random_state=0).reset_index(drop=True)
print(f"{len(acacia):,} presences; sample: {len(sample)}")
sample[["id", "pixel", "geometry"]].to_file(OUT, layer="sample_points", driver="GPKG")

## 1. COS 2023 polygon

No new data or dependency needed: `data/interim/spatial/COS2023v1-S2.gpkg`
already has the "Florestas de acácias" class, already used in
`03_expert_reference/expert_polygons.ipynb` and
`02_citizen_science/acacia_observations.ipynb`. Known limitation
(checked earlier on the full 3 000): only ~17–19% of presences fall
inside one of these polygons, so this rule alone leaves most
observations without a mask.

In [ ]:
buffer = sample.buffer(200)  # generous margin around the sample so the polygon join doesn't miss edge cases
cos_acacia = gpd.read_file(COS, layer="COS2023v1", where="COS23_n4_C = '5.1.1.7'", bbox=tuple(buffer.total_bounds))
mask_cos = gpd.sjoin(sample[["id", "geometry"]], cos_acacia[["geometry"]], predicate="within", how="left")
mask_cos = mask_cos.dropna(subset=["index_right"]).merge(
    cos_acacia.geometry.rename("mask_geometry"), left_on="index_right", right_index=True)
mask_cos = gpd.GeoDataFrame(mask_cos[["id"]], geometry=mask_cos["mask_geometry"], crs=3763)
print(f"{len(mask_cos)}/{len(sample)} sample points fall inside a COS acacia polygon")
if len(mask_cos):
    mask_cos.to_file(OUT, layer="mask_cos", driver="GPKG")

## 2. Region growing on TESSERA embeddings

**Update (2026-11-03): unblocked.** The dead end above was only the
high-level `GeoTessera` class (`fetch_embedding`,
`fetch_mosaic_for_region`, `sample_embeddings_at_points`), which reads
the `cambridge` NPY dataset only — no Portugal coverage, confirmed
directly against the bucket. The library has a separate, lower-level
class, `geotessera.store.GeoTesseraZarr`, that reads the `dclimate`
dataset (the one that actually covers Portugal) straight from its
Zarr/Icechunk store over HTTPS — this is the same store the deleted
RQ1 pilot already used, just for single pixels
(`sample_points`) instead of a patch (`read_patch`). See
[`notebooks/04_satellite/tessera_embeddings.ipynb`](../04_satellite/tessera_embeddings.ipynb)
for the working access method, confirmed with a real patch (21×21 px,
210×210 m, no NaN) around a presence.

Superpixels (candidate 3) can now also use this patch as its input
raster — still not attempted here.

**Rule (my own, not from a paper — see above):** read a 41×41 px
(410×410 m) patch centred on the point; starting from the centre
pixel, flood-fill to 4-connected neighbours whose embedding has
cosine similarity ≥ a threshold to the centre's, up to 20 px (200 m)
from the centre. Threshold and radius are **not grounded in anything
— guesses, to be calibrated by looking at the result**, not a decided
rule.

In [ ]:
import numpy as np
from geotessera.store import GeoTesseraZarr
from rasterio.features import shapes as raster_shapes
from shapely.geometry import shape as shapely_shape
from shapely.ops import unary_union

store = GeoTesseraZarr(cache_dir=ROOT / "data" / "raw" / "satellite" / "tessera_zarr_cache")
YEAR = 2023
SIZE_PX = 41
SIM_THRESHOLD = 0.90
MAX_RADIUS_PX = 20


def grow_region(patch, threshold=SIM_THRESHOLD, max_radius=MAX_RADIUS_PX):
    h, w, _ = patch.shape
    seed = (h // 2, w // 2)
    seed_emb = patch[seed]
    sim = (patch @ seed_emb) / (np.linalg.norm(patch, axis=-1) * np.linalg.norm(seed_emb))
    mask = np.zeros((h, w), dtype=bool)
    mask[seed] = True
    stack = [seed]
    while stack:
        r, c = stack.pop()
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if (0 <= nr < h and 0 <= nc < w and not mask[nr, nc]
                    and max(abs(nr - seed[0]), abs(nc - seed[1])) <= max_radius
                    and sim[nr, nc] >= threshold):
                mask[nr, nc] = True
                stack.append((nr, nc))
    return mask


sample_wgs84 = sample.to_crs(4326)
rows, pixel_counts = [], []
for i, row in sample.iterrows():
    lon, lat = sample_wgs84.geometry.iloc[i].x, sample_wgs84.geometry.iloc[i].y
    patch, transform, crs = store.read_patch(lon=lon, lat=lat, year=YEAR, size_px=SIZE_PX)
    mask = grow_region(patch)
    pixel_counts.append(int(mask.sum()))
    polys = [shapely_shape(geom) for geom, val in raster_shapes(mask.astype("uint8"), mask=mask, transform=transform) if val]
    rows.append({"id": row["id"], "n_pixels": int(mask.sum()), "geometry": unary_union(polys)})

mask_tessera = gpd.GeoDataFrame(rows, crs=crs).to_crs(3763)
mask_tessera.to_file(OUT, layer="mask_tessera", driver="GPKG")
print(f"pixels per mask: min {min(pixel_counts)}, median {int(np.median(pixel_counts))}, max {max(pixel_counts)} "
      f"(1 pixel = seed only; {MAX_RADIUS_PX * 2 + 1}x{MAX_RADIUS_PX * 2 + 1} = hit the window edge)")

**Check one example** before trusting the numbers above: PCA-RGB of
its patch, with the grown region outlined.

In [ ]:
from sklearn.decomposition import PCA

example = int(np.argsort(pixel_counts)[len(pixel_counts) // 2])  # a median-sized one, not a trivial min/max
lon, lat = sample_wgs84.geometry.iloc[example].x, sample_wgs84.geometry.iloc[example].y
patch, transform, crs = store.read_patch(lon=lon, lat=lat, year=YEAR, size_px=SIZE_PX)
mask = grow_region(patch)

rgb = PCA(n_components=3).fit_transform(patch.reshape(-1, patch.shape[-1])).reshape(*patch.shape[:2], 3)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(rgb)
ax.contour(mask, levels=[0.5], colors="red", linewidths=2)
ax.scatter([SIZE_PX // 2], [SIZE_PX // 2], c="red", marker="x", s=80)
ax.set_title(f"id {sample.loc[example, 'id']}: {pixel_counts[example]} pixels "
             f"(threshold {SIM_THRESHOLD}, max radius {MAX_RADIUS_PX} px)")
plt.show()